In [1]:

using Plots, Distributions, JLD, DSP, Parameters, Match, SymDict, NumericalIntegration, Random, MAT, Pkg, CSV, DataFrames, XLSX, Pkg, JLD2, Colors, JLD2,  Makie, CairoMakie, Optim, Serialization, JuMP, Ipopt, ForwardDiff, LinearAlgebra, Wandb, DifferentialEquations, LineSearches, BlackBoxOptim
include("creation_model.jl")
# include("optimisation.jl")
# include("optimisation_constant_inhibition.jl")


spikes_from_aps

## Import experimental data

In [2]:
# data_exp = DataFrame(CSV.File("/Users/nmt121/Desktop/Repositories/Motoneuron_Model/Data_MN_Model/csv_subjects/S5_short_hyst.csv"))
# hysteresis_exp= (data_exp[:,2]./ data_exp[:,1]).-1; #force(derecruitment) - force(recruitment)
# hysteresis_exp

df_hyst = CSV.read("/Users/nmt121/Desktop/Repositories/Motoneuron_Model/Data_MN_Model/hysteresis.csv",DataFrame; header=false);
hysteresis_exp = df_hyst[:, 1];

## Initiliase parameters

In [47]:
######################  S-TYPE ##########################
#########################################################
JLD2.@load "/Users/nmt121/Desktop/Repositories/Motoneuron_Model/Data_MN_Model/regressed_parameter_87neurons_NOIF.jld2" regressed_parameter

nr_s = 87

# Radius and soma lengths
rheobase_S = LinRange(3.5,6.5,nr_s) .+ rand(Normal(0.0,0.05),nr_s)  
rs_S = regressed_parameter[1,:]
ls_S = regressed_parameter[2,:] 
rms_S = regressed_parameter[3,:]
rd_S = regressed_parameter[4,:]
ld_S = regressed_parameter[5,:]
rmd_S = regressed_parameter[6,:]

#  Maximun conductances for each of the ION channels mS/cm^2 
max_gna = 30.0 
max_gkf_S = ones(nr_s)*4 
max_gks_S = regressed_parameter[7,:]
max_gca_S = regressed_parameter[8,:]


# Rate constants of the kinetic processes in ms^-1
αm = 22.0; βm = 13.0
αh_S = ones(nr_s)*0.5
βh_S = ones(nr_s)*4
αn_S = ones(nr_s)*1.5
βn_S = ones(nr_s)*0.1
αq_S = ones(nr_s)*1.5
βq_S = ones(nr_s)*0.025
αp = 0.008
βp= regressed_parameter[9,:]
βp_S = ones(nr_s)*0.014

Vthca_S = regressed_parameter[10,:]
Apeak_S =  regressed_parameter[11,:]
tpeak_S = LinRange(110.,100.,nr_s) 
tetanic_S = LinRange(0.393,0.491,nr_s);


#EXCITATORY PROFILE
amplitude=0.4
tr_length=28000
half_triang = div(tr_length, 2)  
flat_line = fill(-0.1, 1000)
up = range(-0.1, amplitude, length = half_triang + 1)  
down = range(amplitude, -0.1, length = tr_length - half_triang + 1)  
tr = vcat(flat_line, up[2:end], down[2:end], flat_line)
sim_length=30000

#AMPLIFICATION
γa=1.;

#INHIBITION FACTOR (SIZE ORDER)
IF= LinRange(1.,1,nr_s);  #factor for the inhibtion
# EF= LinRange(0.8,1.8, nr_s);

#reversal potentials
EL=0; 
ENa=120; ECa=140; EK=-10; Ee=70; Ei=-16 
#maximun conductances
Ge=0.6; Gi=0.5 ; 
#initial voltages
Vd0 = 0.; 
Vs0 = 0.;
#refractory period
rp=5.0; 
_int(num) = floor(Int, num);
pd=0.6; #Pulse duration [mS]




In [61]:
γh = zeros(nr_s);


In [62]:
γh_initial=γh;

In [63]:
ri = 70.  # Ω.cm 3intracellular resistivity
cm = 1.; #membrane capacitance

MN_list=[];
Vth_S = zeros(nr_s); gls_S = zeros(nr_s); gld_S = zeros(nr_s); gc_S = zeros(nr_s); gna_S = zeros(nr_s); gkf_S = zeros(nr_s); gks_S = zeros(nr_s); gca_S = zeros(nr_s); rn_S = zeros(nr_s); cs_S = zeros(nr_s)  ; cd_S = zeros(nr_s)
for i in 1:nr_s 
    ##assuming that the somatic membrane can be modeled as a cylinder and using the defined parameters from the previous section
    gls_S[i] = ((2*π*rs_S[i]*ls_S[i])/rms_S[i])*1e6  # conductance of the soma for the ith neuron (uS)
    gld_S[i] = ((2*π*rd_S[i]*ld_S[i])/rmd_S[i])*1e6  # # conductance of the dendrite for the ith neuron (uS)
    gc_S[i] = ((2/(((ri*ld_S[i])/(π*rd_S[i]^2))+((ri*ls_S[i])/(π*rs_S[i]^2)))))*1e6    # coupling conductance in uS
    rn_S[i] = 1/(gls_S[i]+((gld_S[i]*gc_S[i])/(gld_S[i]+gc_S[i]))) # input resistance of the neuron in MΩ     
    Vth_S[i] = round(rn_S[i]*rheobase_S[i],digits=2)   # in mV
    cs_S[i] = 2*π*ls_S[i]*rs_S[i]*cm*1e3   #inner capacitance of the soma nF
    cd_S[i] = 2*π*ld_S[i]*rd_S[i]*cm*1e3   #nF
    
    #conductances uS:
    gna_S[i] = 2*π*ls_S[i]*rs_S[i]*max_gna*1e3          
    gkf_S[i] = 2*π*ls_S[i]*rs_S[i]*max_gkf_S[i]*1e3     
    gks_S[i] = 2*π*ls_S[i]*rs_S[i]*max_gks_S[i]*1e3     
    gca_S[i] = 2*π*ld_S[i]*rd_S[i]*max_gca_S[i]*1e3     

    S_MN = Dict(:Cs => cs_S[i], :Cd => cd_S[i], :Irh => rheobase_S[i], :gc => gc_S[i], :gNa => gna_S[i], :gKs => gks_S[i], :gKf => gkf_S[i], :gCa => gca_S[i], :gld => gld_S[i], :gls => gls_S[i], :αp => αp, :βp => βp_S[i], 
        :αq => αq_S[i], :βq => βq_S[i], :αm => αm, :βm => βm, :αh => αh_S[i], :βh => βh_S[i], :αn => αn_S[i], :βn => βn_S[i],:αp => αp, :βp => βp[i],  :Vth => Vth_S[i], :VthCa => Vthca_S[i], :Rn => rn_S[i],
        :Tetanic => tetanic_S[i], :Apeak => Apeak_S[i],:IF => IF[i], :tpeak => tpeak_S[i], :γh=>γh[i] , "type" => "All Type")
 
    MN_list = [MN_list;S_MN];
end
MNs = MN_list;


tspan = 0:0.05:sim_length; #time span of simulation defined by step size and length of simulation // change this to get more or less points in the simulation
m=0.0;h=1.0;n=0.0;q=0.0;p=0.0;m0=0.0;h0=1.0;n0=0.0;q0=0.0;p0=0.0; t0=-1.0; t1=-1.0; SPIKE0=0; SPIKE1=0
pool_state_params = fill((m, h, n, q, p, m0, h0, n0, q0, p0, t0, t1, SPIKE0, SPIKE1), nr_s) 
sim_time = _int(sim_length/0.05)   # because the step size is 0.05ms
spike_arr = zeros((nr_s, sim_time)) #storage
vs_arr = zeros((nr_s, sim_time)); #storage
commoninput = tr; #needed for the run_sim() function


full_input = zeros((nr_s, sim_length))
independantnoise = zeros((nr_s, sim_length))
factor= 0.3 #for creating the inhibitory signal
inh_input=fill(factor*amplitude, sim_length);


## Function optim with constant inhibition

In [10]:
function object_function_constant_inh(γh,i)
    simulated_hysteresis, _ = simulate_hh_hysteresis_constant_inh(γh,i,MN_list)
    ssd = sum((hysteresis_exp[i] .- simulated_hysteresis).^2)
    return ssd 
end



function run_sim_optim_constant_inh(i)
    full_input = commoninput 
    _, vs, _ = ode4(tspan, 0.,0., i, pool_state_params[i], full_input, inh_input, MNs)       
    spikes = spikes_from_aps(vs, i, MNs)       
    spike_arr = reshape(spikes, 1, length(spikes))
    vs_arr[i,:] = vs
    return spike_arr, vs_arr
end



function firing_rates_max(spike_array,i)
    max_rates_mns=[]
    for i in 1: size(spike_array)[1]
        spike_pos=[]
        for j in 1:size(spike_array)[2]
            if spike_array[i,j]==1.0
                push!(spike_pos,j)
            end
        end
        ISI= diff(spike_pos);
        rates= 1 ./(ISI/20000)
        max_rates= maximum(rates)
        max_rates_mns=max_rates    
    end 
    max_rate_mn= max_rates_mns;
    return max_rate_mn;
    end



function normalize_to_range(vector, new_min, new_max)
        min_val = minimum(vector)
        max_val = maximum(vector)
        return ((vector .- min_val) / (max_val - min_val)) * (new_max - new_min) .+ new_min
end



function simulate_hh_hysteresis_constant_inh(γh::Vector,i,MN_list)   
    MN_list[i][:γh] = γh[i]
    MNs= MN_list[i]
    spike_array, _ = run_sim_optim_constant_inh(i);


    amplitude=0.4;
    sim_length=560000 ;
    half_triang = div(sim_length, 2);  
    flat_line = fill(-0.1, 20000);
    up = range(-0.1, amplitude, length = half_triang + 1)  ;
    down = range(amplitude, -0.1, length = sim_length - half_triang + 1);  
    tr = vcat(flat_line, up[2:end], down[2:end], flat_line);
    equiv_force_profile= (tr/maximum(tr))*nr_s

    hysteresis= []

    if any(spike_array[1, :] .== 1)
            
        value_1= equiv_force_profile[findfirst(==(1),spike_array[1,:]),:] #force at the recruitment threshold
        value_2=  equiv_force_profile[findlast(==(1),spike_array[1,:]),:] #force at the derecruitment threshold

        #random simulation example
        values1_example=[35.33365178571428, 34.94564732142857, 35.340254464285714, 37.73275446428572, 37.3828125, 36.383477678571424, 36.32405357142857, 38.22329464285714, 38.00734821428571, 37.36805357142857, 37.3665, 38.56935267857143, 38.60858035714286, 38.03841964285714, 38.49050892857143, 41.16265178571428, 39.40633928571429, 39.89221875, 40.390526785714286, 41.258196428571424, 41.12187053571428, 42.19033928571428, 42.20276785714285, 41.844281249999995, 42.10644642857143, 42.373272321428566, 42.09906696428571, 42.751566964285715, 43.571075892857145, 42.99470089285714, 43.69575, 43.546607142857134, 43.98199553571428, 44.49583928571428, 44.27290178571428, 44.64614732142857, 46.472758928571416, 45.30641517857142, 45.49090178571428, 45.574406249999996, 45.825308035714286, 45.67888392857142, 45.92900892857142, 45.572075892857136, 45.26485714285714, 47.002138392857134, 46.38847767857142, 47.34703125, 47.98438392857142, 47.34470089285714, 48.441522321428565, 48.52463839285714, 48.542504464285706, 47.86941964285713, 48.202660714285706, 49.38842410714285, 47.97855803571428, 49.79701339285713, 48.975562499999995, 48.415888392857134, 49.60437053571429, 49.45600446428571, 50.699638392857146, 49.0408125, 49.93877678571429, 50.46116517857143, 50.811495535714286, 51.27057589285714, 51.037151785714286, 50.19317410714285, 50.14229464285714, 52.10872767857142, 50.775763392857144, 51.20882142857143, 51.26902232142857, 50.350473214285714, 50.76721875, 50.78625, 51.13230803571429, 51.37699553571428, 51.686156249999996, 51.46205357142857, 52.10600892857143, 52.325450892857134, 51.94793303571428, 51.92346428571428, 51.10667410714286]
        values2_example= [30.444950892857136, 29.80876339285714, 30.708669642857142, 32.67199553571429, 32.71355357142857, 31.377870535714283, 31.756941964285716, 33.90125892857143, 33.14661160714285, 32.57179017857143, 32.304964285714284, 33.744348214285715, 33.52335267857143, 32.94387053571428, 34.04651785714285, 36.546214285714285, 34.45937946428572, 35.05944642857143, 35.546879464285716, 36.126749999999994, 35.85798214285714, 37.32494196428571, 37.153660714285714, 36.83634375, 37.1925, 37.52768303571428, 36.71128125, 37.643424107142856, 38.74102232142857, 37.62439285714286, 38.302138392857145, 38.02133035714286, 38.54216517857143, 39.50499107142857, 39.51004017857142, 40.08835714285714, 41.392580357142855, 40.259249999999994, 40.80066964285714, 40.310517857142855, 40.85465625, 40.360232142857136, 40.62472767857143, 40.20798214285714, 39.5686875, 41.52502232142857, 41.63377232142857, 41.425982142857144, 42.50804464285714, 42.11033035714285, 42.725933035714284, 43.075874999999996, 42.886339285714286, 42.71583482142857, 42.979553571428575, 44.36961160714286, 42.52862946428571, 44.14162499999999, 43.67050446428571, 42.65641071428571, 44.18473660714285, 43.78585714285714, 44.77431696428571, 43.55243303571428, 44.629834821428574, 45.07260267857143, 45.174361607142856, 45.125812499999995, 44.68887053571429, 43.878294642857135, 43.60719642857142, 45.535566964285714, 44.37893303571428, 45.51070982142857, 44.77858928571429, 44.59759821428571, 44.34786160714286, 44.892, 45.333214285714284, 45.12464732142857, 45.51575892857143, 45.161933035714284, 45.36350892857143, 45.934834821428566, 44.95569642857143, 44.567303571428575, 43.75672767857142]

        values1_example[i]= value_1[1]
        values2_example[i]= value_2[1]

        simulated_values = vcat(values1_example, values2_example)
        normalized_recr = normalize_to_range(simulated_values, 0, 100)
               
        hysteresis= (normalized_recr[i+10]/normalized_recr[i]).-1

        if normalized_recr[i+10]==normalized_recr[i]
            hysteresis=100 #random high value
        end

    else
        hysteresis=100 #random high value
    end

    max_firing_rates= [] #not relevant now firing_rates_max(spike_array,i) SOLVE THE CASE IN WHICH THERE ARE NO SPIKES
    
    return hysteresis, max_firing_rates
end


# function compute_gradient_manual!(g::Vector{Float64}, γh::Vector{Float64}, i::Int; ε::Float64=1e-3)
#     γh_plus = copy(γh)
#     γh_minus = copy(γh)
    
#     γh_plus[i] += ε
#     γh_minus[i] -= ε

#     f_plus = BigFloat(object_function_constant_inh(γh_plus, i))
#     f_minus = BigFloat(object_function_constant_inh(γh_minus, i))

#     g[i] = (f_plus - f_minus) / (2 * ε)
    

# end

# function gradient_descent_optimization_constant_inh(γh_initial::Vector{Float64}, i::Int, learning_rate::Float64, max_iterations::Int, tolerance::Float64)
    
#     γh = copy(γh_initial) #make a copy of the initial values of the hysteresis profile
#     g = zeros(Float64, length(γh))  # Gradient vector initialization
#     obj_value_prev = BigFloat(object_function_constant_inh(γh, i))  # Initial objective function value

#     for iter in 1:max_iterations #for loop for going through all the iterations for ONE motoneuron
#         fill!(g, 0.0)  # Reset gradient vector
#         compute_gradient_manual!(g, γh, i)  # Compute gradient
        
#         for j in 1:nr_s
#             γh[j] -= learning_rate * g[j] #new value fo the hysteresis profile, doing it for all MNs, but in reality later only taking the one we are interested in 
#         end

#         γh = clamp.(γh, 0.0, 1.0)  # Ensure γh remains within bounds [0, 1]

#         obj_value_new = BigFloat(object_function_constant_inh(γh, i)) # Compute new objective function value for the motoneuron we want

#         if abs(obj_value_new - obj_value_prev) < tolerance #
#             println("Convergence achieved at iteration $iter")
#             break  
#         end
        

#         obj_value_prev = obj_value_new  # Update previous objective function value for next iteration

#         println("Iteration $iter, Objective Function Value: ", obj_value_new)
#     end

#     return γh  
# end

simulate_hh_hysteresis_constant_inh (generic function with 1 method)

## Functions optim not constant inhibition (i.e.: push-pull)

In [17]:
function object_function(γh,i)
    penalty = 0
    penalty_factor = 1e6  # Adjust

    if γh[i] < 0. || γh[i] > 1.
        penalty += penalty_factor * (0 - γh[i])^2
    end

    simulated_hysteresis, max_firing_rates = simulate_hh_hysteresis(γh,i)
    ssd = sum((hysteresis_exp[i] .- simulated_hysteresis).^2)
    
    return ssd + penalty
end


function run_sim_optim(i)

    full_input = commoninput 
    vd, vs, _ = ode4(tspan, 0.,0., i, pool_state_params[i], full_input, full_input*factor, MNs)       
    spikes = spikes_from_aps(vs, i, MNs)       
    spike_arr = reshape(spikes, 1, length(spikes))
    vs_arr[i,:] = vs
    return spike_arr, vs_arr
end


function firing_rates_max(spike_array,i)

        max_rates_mns=[]
    for i in 1: size(spike_array)[1]
        spike_pos=[]
        for j in 1:size(spike_array)[2]
            if spike_array[i,j]==1.0
                push!(spike_pos,j)
            end
        end
        ISI= diff(spike_pos);
        rates= 1 ./(ISI/20000)
        max_rates= maximum(rates)
        max_rates_mns=max_rates
        
    end
    
    max_rate_mn= max_rates_mns;
    
    return max_rate_mn;
    
    end


function simulate_hh_hysteresis(γh::Vector,i)    
    MN_list[i][:γh] = γh[i]

    MNs= MN_list[i]

    spike_array, vs_array = run_sim_optim(i);

    amplitude=0.4;
    sim_length=560000 ;
    half_triang = div(sim_length, 2);  
    flat_line = fill(-0.1, 20000);
    up = range(-0.1, amplitude, length = half_triang + 1)  ;
    down = range(amplitude, -0.1, length = sim_length - half_triang + 1);  
    tr = vcat(flat_line, up[2:end], down[2:end], flat_line);
    equiv_force_profile= (tr/maximum(tr))*nr_s

    hysteresis= []

    value_1= equiv_force_profile[findfirst(==(1),spike_array[1,:]),:] #force at the recruitment threshold
    value_2=  equiv_force_profile[findlast(==(1),spike_array[1,:]),:] #force at the derecruitment threshold

    if value_1 !== nothing
        # Safe to use index
    else
        print(11111111)
    end

    if value_2 !== nothing
        # Safe to use index
    else
        print(222222222)
    end
    hysteresis= (value_2/value_1).-1
    # println(hysteresis)

    max_firing_rates= firing_rates_max(spike_array,i)
    
    return hysteresis, max_firing_rates
end


function compute_gradient_manual!(g::Vector{Float64}, γh::Vector{Float64}, i::Int; ε::Float64=1e-3)
    γh_plus = copy(γh)
    γh_minus = copy(γh)
    
    γh_plus[i] += ε
    γh_minus[i] -= ε

    f_plus = BigFloat(object_function(γh_plus, i))
    f_minus = BigFloat(object_function(γh_minus, i))

    g[i] = (f_plus - f_minus) / (2 * ε)
end

function gradient_descent_optimization(γh_initial::Vector{Float64}, i::Int, learning_rate::Float64, max_iterations::Int, tolerance::Float64)
    
    γh = copy(γh_initial)  # Initial parameters
    g = zeros(Float64, length(γh))  # Gradient vector initialization
    obj_value_prev = BigFloat(object_function(γh, i))  # Initial objective function value

    for iter in 1:max_iterations
        fill!(g, 0.0)  # Reset gradient vector
        compute_gradient_manual!(g, γh, i)  # Compute gradient

        
        for j in 1:length(γh)
            γh[j] -= learning_rate * g[j]
        end

        # Ensure γh remains within bounds [0, 1]
        γh = clamp.(γh, 0.0, 1.0)

        # Compute new objective function value
        obj_value_new = BigFloat(object_function(γh, i))


        if abs(obj_value_new - obj_value_prev) < tolerance
            println("Convergence achieved at iteration $iter")
            break  
        end

        obj_value_prev = obj_value_new  # Update previous objective function value for next iteration

        println("Iteration $iter, Objective Function Value: ", obj_value_new)
    end

    return γh  
end



gradient_descent_optimization (generic function with 2 methods)

## Running optim with manual gradient descent (NOT IDEAL)

In [68]:
optimized_γh=zeros(1,nr_s)
for i in 1:nr_s
    println(i)
    if hysteresis_exp[i]<0
        optimized_γh[1,i]= gradient_descent_optimization_constant_inh(γh_initial,i, 0.001, 100,1e-5)[i]
        println(optimized_γh[1,i])
    else
        optimized_γh[1,i]= γh_initial[i]
        println("No Hysteresis")

    end
end

1
No Hysteresis
2
No Hysteresis
3
No Hysteresis
4
Iteration 1, Objective Function Value: 0.006908094799257031258521433159103253274224698543548583984375
Iteration 2, Objective Function Value: 0.0110332377844162589364263027391643845476210117340087890625
Iteration 3, Objective Function Value: 0.00697786638165286132728937218416831456124782562255859375
Iteration 4, Objective Function Value: 0.007453144861823773024978034840160034946165978908538818359375
Iteration 5, Objective Function Value: 0.01098395257277792226580093171151020214892923831939697265625
Iteration 6, Objective Function Value: 0.01200994942157504984103777445625382824800908565521240234375
Iteration 7, Objective Function Value: 0.00682945788645849920228680929312758962623775005340576171875
Iteration 8, Objective Function Value: 0.01174390317329104642396497837353308568708598613739013671875
Iteration 9, Objective Function Value: 0.0065720203638131614864992258162601501680910587310791015625
Iteration 10, Objective Function Value: 0.00

InterruptException: InterruptException:

## Optim using libraries

In [11]:
###################### BFGS ############################################
########################################################################
function objective_function(γh, i)
    total_error = 0
    hysteresis, _ = simulate_hh_hysteresis_constant_inh(γh, i, MN_list)
    if hysteresis != 100  #  100 is used as a placeholder for invalid results
        error = (hysteresis - hysteresis_exp[i])^2
        total_error += error
    else
        total_error += 1. # Penalize invalid results with a fixed error
    end
    return total_error
end

function julia_objective(x, i)
    x[1] = clamp(x[1], 0.0, 1.0)  # γh is within the bounds
    temporary_γh = copy(γh_initial)
    temporary_γh[i] = x[1]
    ssd = objective_function(temporary_γh, i)
    println("Testing γh[", i, "] = ", temporary_γh[i], " | Error + Penalty: ", ssd)
    return ssd
end

lower_bounds = [0.0]
upper_bounds = [1.0]
outer_optimizer = BFGS(linesearch=HagerZhang()) 
options = Optim.Options(show_every=1, extended_trace=true, f_tol=1e-6, x_tol=1e-6, g_tol=1e-6, iterations=500 )

new_γ = Dict()
for i in 1:87
    if hysteresis_exp[i]<0.5
        result = optimize(x -> julia_objective(x, i), lower_bounds, upper_bounds, [γh_initial[i]], Fminbox(outer_optimizer), options)
        new_γ[i] = (minimizer = result.minimizer, minimum = result.minimum)
        println("Optimized Result for Neuron ", i, ": ", result.minimizer, " with objective value: ", result.minimum)
    else 
        new_γ[i]=0
    end
end


┌ Warning: Initial position cannot be on the boundary of the box. Moving elements to the interior.
│ Element indices affected: [1]
└ @ Optim /Users/nmt121/.julia/packages/Optim/DV9Sq/src/multivariate/solvers/constrained/fminbox.jl:314


Testing γh[7] = 0.010006055454452394 | Error + Penalty: 0.8740562156447196
Testing γh[7] = 0.009993944545547606 | Error + Penalty: 0.8741766810811739
Testing γh[7] = 0.01 | Error + Penalty: 0.8741164510919961
Testing γh[7] = 0.1094700455258961 | Error + Penalty: 0.14223881851323
Testing γh[7] = 0.10945793461699131 | Error + Penalty: 0.14227395627094025
Testing γh[7] = 0.10946399007144371 | Error + Penalty: 0.14223881851323
Testing γh[7] = 0.507326005811671 | Error + Penalty: 0.053951046484914555
Testing γh[7] = 0.5073138949027661 | Error + Penalty: 0.053951046484914555
Testing γh[7] = 0.5073199503572186 | Error + Penalty: 0.053951046484914555
Testing γh[7] = 0.5073252150934672 | Error + Penalty: 0.053951046484914555
Testing γh[7] = 0.5073131041845623 | Error + Penalty: 0.053951046484914555
Testing γh[7] = 0.5073191596390147 | Error + Penalty: 0.053951046484914555
Testing γh[7] = 0.507324926733973 | Error + Penalty: 0.053951046484914555
Testing γh[7] = 0.5073128158250682 | Error + Penal

InterruptException: Error trying to display an error.

In [81]:
global fitness_history = Float64[] 
global smoothed_fitness_values = []

function objective_function(γh, i, MN_list, hysteresis_exp)
    total_error = 0
    hysteresis, _ = simulate_hh_hysteresis_constant_inh(γh, i, MN_list)
    if hysteresis != 100  #  100 is used as a placeholder for invalid results
        error = (hysteresis - hysteresis_exp[i])^2
        total_error += error
    else
        total_error += 1. # Penalize invalid results with a fixed error
    end
    return total_error
end

function moving_average(values, window_size)
    len = length(values)
    if len < window_size
        return values
    else
        smoothed = [mean(values[max(1, i-window_size+1):i]) for i in 1:len]
        return smoothed
    end
end

function should_stop(fitness_history, threshold=1e-6, window=10)
    length(fitness_history) > window && var(fitness_history[end-window:end]) < threshold
end


function wrap_objective(x, i, γh_initial, MN_list, hysteresis_exp)
    x1 = clamp(x[1], 0.0, 1.0)
    temporary_γh = copy(γh_initial)
    temporary_γh[i] = x1
    ssd = objective_function(temporary_γh, i, MN_list, hysteresis_exp)
    push!(fitness_history, ssd)
    smoothed_fitness = moving_average(fitness_history, 10)  # Smoothing over the last 10 entries
    push!(smoothed_fitness_values, smoothed_fitness)
    # println("Iteration $(length(fitness_history)): Testing γh[", i, "] = ", x1, " | Error + Penalty: ", ssd)
    return ssd
end

##################### INIT OPTIM #####################
results = Dict()
for i in 16
    println("Checking neuron ", i)

    if hysteresis_exp[i] < 0.5
        fitness_history = []  
        result = bboptimize(x -> wrap_objective(x, i, γh_initial, MN_list, hysteresis_exp); SearchRange = [(0.0, 1.0)], Method = :adaptive_de_rand_1_bin_radiuslimited, MaxTime = 200.,MutationRate = 0.7,CrossoverRate = 0.8)
        
        best_params = best_candidate(result) #takes the best value found
        best_fitness_val = best_fitness(result) #takes best fitness value found
        results[i] = (minimizer = best_params, minimum = best_fitness_val)

        println("Optimized Result for Neuron ", i, ": ", best_params, " with objective value: ", best_fitness_val)

        # p = Plots.plot(smoothed_fitness_values[end], title="Optimization Progress for Neuron " * string(i), xlabel="Iteration", ylabel="Smoothed Fitness", legend=false)
        # display(p)

        if should_stop(fitness_history, 1e-6, 10)
            println("Stopping Optimization: Convergence criteria met.")
            break
        end

        
    else #for the first motoneurons that present 0 hysteresis
        println("Neuron ", i, " did not meet the conditions")
        results[i] = 0
    end
end


Checking neuron 16
Starting optimization with optimizer DiffEvoOpt{FitPopulation{Float64}, RadiusLimitedSelector, BlackBoxOptim.AdaptiveDiffEvoRandBin{3}, RandomBound{ContinuousRectSearchSpace}}
0.00 secs, 0 evals, 0 steps
5.80 secs, 2 evals, 1 steps, fitness=0.304452942
11.51 secs, 4 evals, 2 steps, fitness=0.291641703
17.16 secs, 6 evals, 3 steps, improv/step: 0.333 (last = 1.0000), fitness=0.000666720
22.88 secs, 8 evals, 4 steps, improv/step: 0.500 (last = 1.0000), fitness=0.000666720
28.64 secs, 10 evals, 5 steps, improv/step: 0.600 (last = 1.0000), fitness=0.000666720
34.77 secs, 12 evals, 6 steps

In [73]:
# results[11].minimizer[1]

KeyError: KeyError: key 11 not found

In [ ]:
γn_new= zeros(1,nr_s)

for i in 1:nr_s
    if hysteresis_exp[i]<0.5
        γn_new[i]= new_γ[i].minimizer[1]
    else
        γn_new[i]=0
    end
end

In [ ]:
[0,0,0,0,0,0,0.9740209113675593,0.21264078280009352,0.14472313210810428,0.10958682060369039,0.06097812794485675,0.05686518952310181,0.17161877574457848,0.041056424739357715,0.07138663069836941]